# submission OTTO

Ноутбук подключает модуль `recommender.py`, загружает готовые Item2Vec и CatBoost, обрабатывает **все** сессии `test.jsonl` и создаёт `/kaggle/working/submission.csv`. Для каждой сессии сохраняются три строки: `clicks`, `carts`, `orders`, 20 уникальных товаров в каждой.

Для отправки используется текущий сохранённый комплект моделей, он обучен на событиях до границы локальной валидации. (В будущем можно обучить на всех событиях, тем более что более свежие данные важнее прошлых)


In [1]:
!pip install faiss-cpu --quiet

In [2]:
import csv
import json
import sys
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import ujson

TEST_PATH = Path('/kaggle/input/competitions/otto-recommender-system/test.jsonl')
MODELS_DIR = Path('/kaggle/input/notebooks/igorvorobev7892/recproj2-item2vec-training')
CATBOOST_DIR = Path('/kaggle/input/notebooks/igorvorobev7892/recproj2-catboost-training')
CANDIDATE_MODULE_PATH = Path('/kaggle/input/notebooks/igorvorobev7892/recproj2-candidates/recproj2_candidates.py')
RECOMMENDER_MODULE_PATH = Path('/kaggle/input/notebooks/igorvorobev7892/recproj2-recommender/recommender.py')

OUT_DIR = Path('/kaggle/working')
SUBMISSION_PATH = OUT_DIR / 'submission.csv'
BATCH_SESSIONS = 250
THREAD_COUNT = 4

sys.path.insert(0, str(CANDIDATE_MODULE_PATH.parent))
sys.path.insert(0, str(RECOMMENDER_MODULE_PATH.parent))

from recommender import EVENT_TYPES, Item2VecCatboostRecommender


## Формирование submission.csv

CatBoost оценивает кандидатов батчами. Три потока предсказаний (по потоку на тип события, чтобы постоянно не перезагружать в память Item2Vec модель для текущего типа события) затем объединяются в порядке сессий исходного файла; промежуточные CSV удаляются автоматически.


In [3]:
def iter_session_batches(test_path, batch_sessions=250):
    batch = []
    with open(test_path) as test_file:
        for line in test_file:
            batch.append(ujson.loads(line))
            if len(batch) >= batch_sessions:
                yield batch
                batch = []
    if batch:
        yield batch


def write_submission(recommender, test_path, output_path, batch_sessions=250):
    """Три прохода по test и порционная запись в итоговый CSV."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with TemporaryDirectory(prefix='recproj2_predictions_', dir=output_path.parent) as directory:
        directory = Path(directory)
        part_paths = []
        try:
            for event_type in EVENT_TYPES:
                print(f'{event_type}: загрузка модели и построение индекса.', flush=True)
                recommender.load_event_type(event_type)
                part_path = directory / f'{event_type}.csv'
                part_paths.append(part_path)
                processed = 0
                next_report = 50_000
                with open(part_path, 'w', newline='') as file:
                    writer = csv.writer(file)
                    for batch in iter_session_batches(test_path, batch_sessions):
                        predictions = recommender.predict_batch(batch, event_type)
                        writer.writerows(
                            (f"{session['session']}_{event_type}", ' '.join(labels))
                            for session, labels in zip(batch, predictions, strict=True)
                        )
                        processed += len(batch)
                        if processed >= next_report:
                            print(f'{event_type}: {processed:,} сессий.', flush=True)
                            next_report += 50_000
                print(f'{event_type}: готово, {processed:,} сессий.', flush=True)
                recommender.unload_item2vec()
        finally:
            recommender.unload_item2vec()

        session_count = 0
        row_count = 0
        combined_path = directory / 'submission.csv'
        with ExitStack() as stack:
            readers = [
                csv.reader(stack.enter_context(open(path, newline='')))
                for path in part_paths
            ]
            file = stack.enter_context(open(combined_path, 'w', newline=''))
            writer = csv.writer(file)
            writer.writerow(['session_type', 'labels'])
            for session_rows in zip(*readers, strict=True):
                writer.writerows(session_rows)
                session_count += 1
                row_count += len(session_rows)

        combined_path.replace(output_path)

    summary = {
        'sessions': session_count,
        'rows': row_count,
        'submission_path': str(output_path),
        'features': len(recommender.feature_cols),
    }
    with open(output_path.with_name('submission_summary.json'), 'w') as file:
        json.dump(summary, file, ensure_ascii=False, indent=2)
    print(f"Готово: {summary['sessions']:,} сессий, {summary['rows']:,} строк.", flush=True)
    return summary


In [4]:
recommender = Item2VecCatboostRecommender(
    models_dir=MODELS_DIR,
    catboost_dir=CATBOOST_DIR,
    thread_count=THREAD_COUNT,
)

summary = write_submission(
    recommender,
    test_path=TEST_PATH,
    output_path=SUBMISSION_PATH,
    batch_sessions=BATCH_SESSIONS,
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


clicks: загрузка модели и построение индекса.
clicks: 50,000 сессий.
clicks: 100,000 сессий.
clicks: 150,000 сессий.
clicks: 200,000 сессий.
clicks: 250,000 сессий.
clicks: 300,000 сессий.
clicks: 350,000 сессий.
clicks: 400,000 сессий.
clicks: 450,000 сессий.
clicks: 500,000 сессий.
clicks: 550,000 сессий.
clicks: 600,000 сессий.
clicks: 650,000 сессий.
clicks: 700,000 сессий.
clicks: 750,000 сессий.
clicks: 800,000 сессий.
clicks: 850,000 сессий.
clicks: 900,000 сессий.
clicks: 950,000 сессий.
clicks: 1,000,000 сессий.
clicks: 1,050,000 сессий.
clicks: 1,100,000 сессий.
clicks: 1,150,000 сессий.
clicks: 1,200,000 сессий.
clicks: 1,250,000 сессий.
clicks: 1,300,000 сессий.
clicks: 1,350,000 сессий.
clicks: 1,400,000 сессий.
clicks: 1,450,000 сессий.
clicks: 1,500,000 сессий.
clicks: 1,550,000 сессий.
clicks: 1,600,000 сессий.
clicks: 1,650,000 сессий.
clicks: готово, 1,671,803 сессий.
carts: загрузка модели и построение индекса.
carts: 50,000 сессий.
carts: 100,000 сессий.
carts: 150,

In [5]:
display(pd.read_csv(SUBMISSION_PATH, nrows=6))

,session_type,labels
0,12899779_clicks,59625 375825 1231087 1263009 371210 960385 147...
1,12899779_carts,59625 1022566 1025795 1043508 1083665 1111967 ...
2,12899779_orders,59625 1022566 1025795 1040894 1083665 1125638 ...
3,12899780_clicks,1142000 736515 582732 973453 889686 1515511 48...
4,12899780_carts,1142000 582732 973453 1365638 1515511 50708 88...
5,12899780_orders,1142000 736515 582732 973453 889686 1586171 10...
